In [0]:
catalog = "movie_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Mapeamento dos caminhos no Volume (criado no notebook 00, schema landing)
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"

# 2. Leitura pura (sem transformações de negócio)
df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=True)
df_financials_raw = spark.read.csv(path_financials, header=True, inferSchema=True)
df_info_raw = spark.read.csv(path_info, header=True, inferSchema=True)
df_metrics_raw = spark.read.csv(path_metrics, header=True, inferSchema=True)
df_reviews_raw = spark.read.csv(path_reviews, header=True, inferSchema=True)

In [0]:
# 3. Gravação com adição do timestamp no momento da escrita
df_credits_and_tags_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags")

df_financials_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_financials")

df_info_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_info")

df_metrics_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_metrics")

df_reviews_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_reviews")

id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_timestamp
293660,72.735,7.606,28894,8.0,1270339,2026-10-06T12:23:09.081Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-06T12:23:09.081Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-06T12:23:09.081Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-06T12:23:09.081Z
271110,70.741,7.4,21541,7.8,947222,2026-10-06T12:23:09.081Z


In [0]:
import json
import requests
from datetime import datetime

# 1. Configuração dos parâmetros (Formato exigido pelo BCB: MM-DD-YYYY)
data_inicio = "09-30-2026"
data_fim = "10-06-2026"

# 2. Montagem da URL com o endpoint do BCB
url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&"
    f"$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# 3. Requisição HTTP (sem necessidade de headers de autenticação)
response = requests.get(url)
response.raise_for_status()
data_json = response.json()

# 4. Extração da lista de cotações contida no OData ('value')
cotacao_dolar = data_json.get("value", [])

# 5. Salvando na Landing Zone (Volume do Unity Catalog)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
file_path = f"{landing_path}/ptax_{timestamp}.json"

# Grava os registros brutos em formato JSON
dbutils.fs.put(file_path, json.dumps(cotacao_dolar), overwrite=True)

Wrote 302 bytes.


True

In [0]:
from pyspark.sql.functions import current_timestamp

path_cotacao_dolar = f"{landing_path}/ptax_20261006_130333.json"
df_cotacao_dolar_raw = spark.read.option("multiLine", "true").json(path_cotacao_dolar)

df_cotacao_dolar_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_cotacao_dolar")

display(df_cotacao_dolar_raw)

cotacaoCompra,dataHoraCotacao
5.1803,2026-09-30 13:11:44.783262
5.2073,2026-10-01 13:10:35.4469
5.2232,2026-10-02 13:03:16.256632
4.9853,2026-10-05 13:07:36.558602
